# AI Project 2025/2026 - KNN Implementation


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error, classification_report, accuracy_score, f1_score

# Configurar precisão de exibição
pd.options.display.float_format = '{:.4f}'.format


## 1. Carregamento dos Dados

In [2]:
df = pd.read_csv('../employee_data/employee_data.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,34,No,Travel_Frequently,702,Research & Development,16,4,Life Sciences,1,838,...,3,80,0,6,3,3,5,2,1,3
1,38,No,Travel_Rarely,833,Research & Development,18,3,Medical,1,1766,...,3,80,1,15,2,3,1,0,1,0
2,51,No,Travel_Rarely,833,Research & Development,1,3,Life Sciences,1,353,...,2,80,0,1,0,2,1,0,0,0
3,60,No,Travel_Rarely,1179,Sales,16,4,Marketing,1,732,...,4,80,0,10,1,3,2,2,2,2
4,23,No,Travel_Rarely,571,Research & Development,12,2,Other,1,1982,...,3,80,0,5,6,4,5,2,1,4


## 2. Preprocessamento

In [3]:
# Identificação de colunas
target_reg = 'MonthlyIncome'
target_clf = 'Attrition'

# Colunas a remover
drop_features = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df_clean = df.drop(columns=drop_features)

# Definir features numéricas e categóricas
numeric_features = df_clean.select_dtypes(include=['int64', 'float64']).drop(columns=[target_reg]).columns.tolist()
categorical_features = df_clean.select_dtypes(include=['object']).drop(columns=[target_clf]).columns.tolist()

# Transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 3. Regressão com KNN: Prever MonthlyIncome

In [4]:
X_reg = df_clean.drop(columns=[target_reg, target_clf])
y_reg = df_clean[target_reg]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

param_grid_reg = {
    'regressor__n_neighbors': [3, 5, 7, 9, 11],
    'regressor__weights': ['uniform', 'distance']
}

pipe_reg = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', KNeighborsRegressor())])
grid_reg = GridSearchCV(pipe_reg, param_grid_reg, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_reg.fit(X_train_reg, y_train_reg)

y_pred_reg = grid_reg.predict(X_test_reg)
rmse = root_mean_squared_error(y_test_reg, y_pred_reg)
print(f"Melhor RMSE (CV): {-grid_reg.best_score_:.4f}")
print(f"KNN Regressor RMSE no Teste: {rmse:.4f}")

Melhor RMSE (CV): 2402.1698
KNN Regressor RMSE no Teste: 2273.8348


## 4. Classificação com KNN: Prever Attrition

In [5]:
X_clf = df_clean.drop(columns=[target_clf])
y_clf = df_clean[target_clf].map({'Yes': 1, 'No': 0})

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

param_grid_clf = {
    'classifier__n_neighbors': [3, 5, 7, 9, 11],
    'classifier__weights': ['uniform', 'distance']
}

pipe_clf = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', KNeighborsClassifier())
])

grid_clf = GridSearchCV(pipe_clf, param_grid_clf, cv=5, scoring='f1', n_jobs=-1)
grid_clf.fit(X_train_clf, y_train_clf)

y_pred_clf = grid_clf.predict(X_test_clf)
print("--- Melhor KNN Classifier ---")
print(f"Melhores Parâmetros: {grid_clf.best_params_}")
print(f"Accuracy: {accuracy_score(y_test_clf, y_pred_clf):.4f}")
print(f"F1-Score: {f1_score(y_test_clf, y_pred_clf):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test_clf, y_pred_clf, digits=4))

--- Melhor KNN Classifier ---
Melhores Parâmetros: {'classifier__n_neighbors': 3, 'classifier__weights': 'uniform'}
Accuracy: 0.8240
F1-Score: 0.2143

Relatório de Classificação:
              precision    recall  f1-score   support

           0     0.8547    0.9524    0.9009       210
           1     0.3750    0.1500    0.2143        40

    accuracy                         0.8240       250
   macro avg     0.6149    0.5512    0.5576       250
weighted avg     0.7779    0.8240    0.7910       250



## 5. Exportação dos Modelos Finais

In [6]:
with open('Group04_KNN_pipeline_regression.pkl', 'wb') as f:
    pickle.dump(grid_reg.best_estimator_, f)

with open('Group04_KNN_pipeline_classification.pkl', 'wb') as f:
    pickle.dump(grid_clf.best_estimator_, f)

print("Modelos KNN exportados com sucesso!")

Modelos KNN exportados com sucesso!
